# 5G-over-SDR Proof-of-Concept - KPI Pipeline

Reproducible pipeline that turns the **raw measurement files** of two 5G SA / USRP B205-mini
test campaigns into tidy CSVs and publication-grade (vector PDF) figures for the ICASC2026 paper.

**Scientific-integrity rule:** every plotted value comes from a real measurement file.
Nothing is invented. Where a value is absent it is printed as `MISSING - not plotted`.

**Two campaigns**

| Run | Type | Band | Source |
|-----|------|------|--------|
| `20260505_162512` | coax SDR-to-SDR | n78 3748.8 MHz, 24 PRB, SCS 30 kHz | tidy CSVs + REPORT.md |
| `20260325_000034` | over-the-air SDR | n78 3604.8 MHz, 24 PRB, SCS 30 kHz | raw ping .txt + iperf3 .json |

Runs top-to-bottom on Windows, Python 3.14, pandas + numpy + matplotlib (no seaborn, no pyarrow).

In [ ]:
# --- Imports and global configuration ---
import os, re, json, glob, shutil
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")              # headless, deterministic
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator

# ---- Output directories (define BASE first, everything else is relative to it) ----
# BASE = the repo root (this notebook lives in <BASE>/notebook/).
BASE     = os.path.abspath(os.path.join(os.getcwd(), ".."))
DATA_OUT = os.path.join(BASE, "data")
FIG_OUT  = os.path.join(BASE, "figures")
PAPER_FIG_OUT = os.path.join(BASE, "paper", "figures")
for d in (DATA_OUT, FIG_OUT, PAPER_FIG_OUT):
    os.makedirs(d, exist_ok=True)

# ---- Source directories (the ONLY ground truth) ----
# Both raw campaigns are vendored inside the repo under raw_measurements/, so the
# notebook is fully self-contained and reproducible on any machine / any clone.
RUN1 = os.path.join(BASE, "raw_measurements", "20260505_162512_coax")            # coax
RUN2_RAW = os.path.join(BASE, "raw_measurements", "20260325_000034_ota", "raw")  # OTA

# ---- IEEE / colorblind-safe figure style ----
# Okabe-Ito blue (#0072B2) + orange (#E69F00): CVD-safe (validated dE ~110).
# Colour is ALWAYS paired with a distinct linestyle + marker so the figures
# also read correctly in grayscale and for low-contrast (orange) rendering.
C_DL, C_UL = "#0072B2", "#E69F00"     # DL = blue, UL = orange
C_REF = "#999999"
plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 120,
    "font.family": "DejaVu Sans",
    "font.size": 9,
    "axes.titlesize": 9,
    "axes.labelsize": 9,
    "legend.fontsize": 8,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "axes.grid": True,
    "grid.color": "#dddddd",
    "grid.linewidth": 0.6,
    "axes.axisbelow": True,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "pdf.fonttype": 42,               # editable TrueType text in the vector PDF
    "ps.fonttype": 42,
})

# Provenance registry: filled as each figure/CSV is produced.
PROV = []   # list of dicts: output, sources[], transformation
def register(output, sources, transformation):
    PROV.append({"output": output,
                 "sources": sources if isinstance(sources, list) else [sources],
                 "transformation": transformation})

def save_fig(fig, name):
    """Save a figure as vector PDF into figures/ and copy into paper/figures/."""
    p1 = os.path.join(FIG_OUT, name)
    fig.savefig(p1, bbox_inches="tight")
    p2 = os.path.join(PAPER_FIG_OUT, name)
    shutil.copyfile(p1, p2)
    plt.close(fig)
    print(f"  saved  {p1}")
    print(f"  copied {p2}")

MISSING = []  # human-readable notes for anything absent
print("BASE =", BASE)
print("Setup OK. pandas", pd.__version__, "| numpy", np.__version__, "| matplotlib", matplotlib.__version__)


## Raw-file parsers

Small, explicit parsers. `ping` .txt -> per-packet RTT list + the summary line
(`rtt min/avg/max/mdev`). iperf3 UDP .json -> the receiver `end.sum`
(`jitter_ms`, `lost_percent`, `bits_per_second`).

In [ ]:
# --- Parser: ping (iputils) text output ---
_PKT_RE = re.compile(r"time=([\d.]+)\s*ms")
_SUM_RE = re.compile(r"rtt min/avg/max/mdev = ([\d.]+)/([\d.]+)/([\d.]+)/([\d.]+)")
_LOSS_RE = re.compile(r"(\d+)%\s*packet loss")

def parse_ping(path):
    """Return dict: rtts(list), min, avg, max, mdev, loss_pct, n."""
    with open(path, "r", encoding="utf-8", errors="ignore") as fh:
        txt = fh.read()
    rtts = [float(x) for x in _PKT_RE.findall(txt)]
    m = _SUM_RE.search(txt)
    summ = dict(min=float(m.group(1)), avg=float(m.group(2)),
                max=float(m.group(3)), mdev=float(m.group(4))) if m else {}
    lm = _LOSS_RE.search(txt)
    loss = float(lm.group(1)) if lm else np.nan
    return {"rtts": rtts, "loss_pct": loss, "n": len(rtts), **summ}

def parse_iperf_udp(path):
    """Return dict from iperf3 UDP JSON end.sum: jitter_ms, lost_percent, bps, packets, seconds."""
    with open(path, "r", encoding="utf-8", errors="ignore") as fh:
        d = json.load(fh)
    s = d["end"]["sum"]
    return {"jitter_ms": s.get("jitter_ms"), "lost_percent": s.get("lost_percent"),
            "bits_per_second": s.get("bits_per_second"), "packets": s.get("packets"),
            "seconds": s.get("seconds")}

# quick self-test on one file of each type
_t = parse_ping(os.path.join(RUN2_RAW, "ping_distribution_1000.txt"))
print("ping_distribution_1000:", _t["n"], "pkts, summary min/avg/max/mdev =",
      _t["min"], _t["avg"], _t["max"], _t["mdev"], "| loss", _t["loss_pct"], "%")
_j = parse_iperf_udp(os.path.join(RUN2_RAW, "iperf3_udp_drone_ul_10pps.json"))
print("drone_ul_10pps:", _j)


## Figure 1 - TCP throughput time series (coax run, 600 s)

Source: `tcp_dl_timeseries.csv`, `tcp_ul_timeseries.csv` (run `20260505_162512`).
Tidy copies are written to `data/`. Mean lines annotated.

In [ ]:
# --- Load + tidy the two TCP time-series CSVs ---
dl = pd.read_csv(os.path.join(RUN1, "tcp_dl_timeseries.csv"))
ul = pd.read_csv(os.path.join(RUN1, "tcp_ul_timeseries.csv"))

def tidy_ts(df, direction):
    out = pd.DataFrame({
        "t_start_s": df["t0"].astype(float),
        "t_end_s":   df["t1"].astype(float),
        "throughput_mbps": df["mbps"].astype(float),
        "retransmits": df["retx"].astype(int),
        "cwnd_kb": df["cwnd_kb"].astype(float),
    })
    out.insert(0, "direction", direction)
    return out

tcp_dl = tidy_ts(dl, "DL")
tcp_ul = tidy_ts(ul, "UL")
tcp_dl.to_csv(os.path.join(DATA_OUT, "tcp_dl_timeseries.csv"), index=False)
tcp_ul.to_csv(os.path.join(DATA_OUT, "tcp_ul_timeseries.csv"), index=False)
register("data/tcp_dl_timeseries.csv", ["20260505_162512/tcp_dl_timeseries.csv"],
         "Renamed columns to tidy schema (t0->t_start_s, t1->t_end_s, mbps->throughput_mbps); added direction=DL.")
register("data/tcp_ul_timeseries.csv", ["20260505_162512/tcp_ul_timeseries.csv"],
         "Renamed columns to tidy schema; added direction=UL.")

dl_mean, ul_mean = tcp_dl["throughput_mbps"].mean(), tcp_ul["throughput_mbps"].mean()

fig, ax = plt.subplots(figsize=(5.4, 2.7))
ax.plot(tcp_dl["t_start_s"], tcp_dl["throughput_mbps"], color=C_DL, lw=1.4,
        ls="-",  marker="o", ms=2.5, label="DL (per 5 s)")
ax.plot(tcp_ul["t_start_s"], tcp_ul["throughput_mbps"], color=C_UL, lw=1.2,
        ls="-",  marker="s", ms=2.5, alpha=0.9, label="UL (per 5 s)")
ax.axhline(dl_mean, color=C_DL, ls="--", lw=1.2)
ax.axhline(ul_mean, color=C_UL, ls=":",  lw=1.4)
ax.text(602, dl_mean, f" DL mean {dl_mean:.2f}", color=C_DL, va="center", fontsize=7.5)
ax.text(602, ul_mean-0.35, f" UL mean {ul_mean:.2f}", color=C_UL, va="center", fontsize=7.5)
ax.set_xlabel("Time (s)"); ax.set_ylabel("TCP throughput (Mbit/s)")
ax.set_xlim(0, 600); ax.set_ylim(0, None)
ax.legend(loc="upper left", framealpha=0.9)
save_fig(fig, "fig_tcp_timeseries.pdf")
register("figures/fig_tcp_timeseries.pdf",
         ["20260505_162512/tcp_dl_timeseries.csv", "20260505_162512/tcp_ul_timeseries.csv"],
         "Per-5s throughput vs time, both directions; horizontal lines at series means.")

def stats(series):
    s = series.astype(float)
    return dict(min=s.min(), mean=s.mean(), max=s.max(), n=len(s))
KPI = {}
KPI["TCP DL throughput (Mbit/s)"] = stats(tcp_dl["throughput_mbps"])
KPI["TCP UL throughput (Mbit/s)"] = stats(tcp_ul["throughput_mbps"])
print("DL", KPI["TCP DL throughput (Mbit/s)"])
print("UL", KPI["TCP UL throughput (Mbit/s)"])


## Figure 2 - UDP achieved-vs-target throughput (coax run)

Source: `udp_dl_sweep.csv`, `udp_ul_sweep.csv` (run `20260505_162512`).
Each point labelled with its measured loss % and jitter (ms). Grey dashed line = ideal `achieved = target`.

In [ ]:
# --- UDP sweep: tidy + plot achieved vs target with loss/jitter labels ---
udl = pd.read_csv(os.path.join(RUN1, "udp_dl_sweep.csv"))
uul = pd.read_csv(os.path.join(RUN1, "udp_ul_sweep.csv"))

def tidy_sweep(df, direction):
    out = df.rename(columns={"target_mbps":"target_mbps","mbps":"achieved_mbps",
                             "jitter_ms":"jitter_ms","lost_pct":"loss_pct"}).copy()
    out.insert(0, "direction", direction)
    keep = ["direction","target_mbps","achieved_mbps","jitter_ms","loss_pct"]
    if "lost" in out.columns:  keep.append("lost")
    if "total" in out.columns: keep.append("total")
    return out[keep]

udp_dl = tidy_sweep(udl, "DL")
udp_ul = tidy_sweep(uul, "UL")
udp_all = pd.concat([udp_dl, udp_ul], ignore_index=True)
udp_all.to_csv(os.path.join(DATA_OUT, "udp_sweep.csv"), index=False)
register("data/udp_sweep.csv",
         ["20260505_162512/udp_dl_sweep.csv", "20260505_162512/udp_ul_sweep.csv"],
         "Concatenated DL+UL sweeps; renamed mbps->achieved_mbps, lost_pct->loss_pct; added direction column.")

# NOTE: DL has only 4 measured target rates and UL only 5 - these are discrete
# operating points from separate iperf3 runs, not a continuous sweep. We deliberately
# plot MARKERS ONLY (no connecting line between points) so the figure never implies
# interpolated/measured-in-between values that do not exist in the data.
fig, ax = plt.subplots(figsize=(5.4, 2.9))
lim = max(udp_all["target_mbps"].max(), udp_all["achieved_mbps"].max()) * 1.08
ax.plot([0, lim], [0, lim], color=C_REF, ls="--", lw=1.0, zorder=1, label="ideal (achieved = target)")
ax.scatter(udp_dl["target_mbps"], udp_dl["achieved_mbps"], color=C_DL, marker="o", s=55,
           zorder=3, edgecolor="white", linewidth=0.7, label=f"DL (n={len(udp_dl)} points)")
ax.scatter(udp_ul["target_mbps"], udp_ul["achieved_mbps"], color=C_UL, marker="s", s=55,
           zorder=3, edgecolor="white", linewidth=0.7, label=f"UL (n={len(udp_ul)} points)")
for _, r in udp_dl.iterrows():
    ax.annotate(f"{r['loss_pct']:.2f}% loss\n{r['jitter_ms']:.2f} ms",
                (r["target_mbps"], r["achieved_mbps"]), textcoords="offset points",
                xytext=(7, -20), fontsize=6.5, color=C_DL)
for _, r in udp_ul.iterrows():
    ax.annotate(f"{r['jitter_ms']:.2f} ms\n{r['loss_pct']:.2f}% loss",
                (r["target_mbps"], r["achieved_mbps"]), textcoords="offset points",
                xytext=(7, 6), fontsize=6.5, color=C_UL)
ax.set_xlabel("Target rate (Mbit/s)"); ax.set_ylabel("Achieved rate (Mbit/s)")
ax.set_xlim(0, lim); ax.set_ylim(0, lim)
ax.legend(loc="upper left", framealpha=0.9)
save_fig(fig, "fig_udp_sweep.pdf")
register("figures/fig_udp_sweep.pdf",
         ["20260505_162512/udp_dl_sweep.csv", "20260505_162512/udp_ul_sweep.csv"],
         "Achieved vs target rate, DL & UL, plotted as discrete markers only (no connecting "
         "line between points, since each direction has few discrete operating points and no "
         "in-between values were measured); per-point loss% and jitter labels; ideal y=x reference.")

KPI["UDP DL jitter (ms)"] = stats(udp_dl["jitter_ms"])
KPI["UDP DL loss (%)"]    = stats(udp_dl["loss_pct"])
KPI["UDP UL jitter (ms)"] = stats(udp_ul["jitter_ms"])
KPI["UDP UL loss (%)"]    = stats(udp_ul["loss_pct"])
print(udp_all.to_string(index=False))


## Figure 3 - Idle ICMP RTT distribution / CDF (OTA run)

Source: `ping_distribution_1000.txt` (run `20260325_000034`, 1000 packets @ 1 pps).
Empirical CDF of per-packet RTT with min / avg / max markers from the ping summary line.

In [ ]:
# --- Idle RTT CDF from 1000-packet ping ---
pd1000 = parse_ping(os.path.join(RUN2_RAW, "ping_distribution_1000.txt"))
rtts = np.array(pd1000["rtts"], dtype=float)
pd.DataFrame({"icmp_seq": np.arange(1, len(rtts)+1), "rtt_ms": rtts}).to_csv(
    os.path.join(DATA_OUT, "ping_distribution_1000.csv"), index=False)
register("data/ping_distribution_1000.csv", ["20260325_000034/raw/ping_distribution_1000.txt"],
         "Extracted per-packet 'time=<x> ms' values (1000 rows).")

xs = np.sort(rtts)
ys = np.arange(1, len(xs)+1) / len(xs)
fig, ax = plt.subplots(figsize=(5.0, 2.8))
ax.step(xs, ys, where="post", color=C_DL, lw=1.6, label=f"empirical CDF (n={len(xs)})")
for val, lab, c, ls in [(pd1000["min"], f"min {pd1000['min']:.2f}", C_UL, ":"),
                        (pd1000["avg"], f"avg {pd1000['avg']:.2f}", "#333333", "--"),
                        (pd1000["max"], f"max {pd1000['max']:.2f}", C_UL, ":")]:
    ax.axvline(val, color=c, ls=ls, lw=1.2)
    ax.text(val, 0.05, " "+lab, rotation=90, va="bottom", ha="left", fontsize=7, color=c)
ax.set_xlabel("ICMP RTT (ms)"); ax.set_ylabel("Cumulative probability")
ax.set_ylim(0, 1.02); ax.legend(loc="lower right", framealpha=0.9)
save_fig(fig, "fig_ping_rtt.pdf")
register("figures/fig_ping_rtt.pdf", ["20260325_000034/raw/ping_distribution_1000.txt"],
         "Empirical CDF of 1000 per-packet RTTs; vertical markers at summary min/avg/max.")

KPI["Idle RTT (ms) [1000 pkt]"] = dict(min=pd1000["min"], mean=pd1000["avg"], max=pd1000["max"], n=pd1000["n"])
print("Idle RTT min/avg/max/mdev =", pd1000["min"], pd1000["avg"], pd1000["max"], pd1000["mdev"],
      "| loss", pd1000["loss_pct"], "%")


## Figure 4 - RTT vs ICMP packet size (OTA run)

Source: `ping_size_{20,64,128,256,512,1024,1400}B.txt` (run `20260325_000034`).
Marker = avg RTT (from each file's summary line); whiskers = min..max.

In [ ]:
# --- RTT vs packet size ---
sizes = [20, 64, 128, 256, 512, 1024, 1400]
rows = []
for b in sizes:
    fp = os.path.join(RUN2_RAW, f"ping_size_{b}B.txt")
    if not os.path.exists(fp):
        MISSING.append(f"ping_size_{b}B.txt absent -> size {b} B not plotted")
        continue
    s = parse_ping(fp)
    rows.append(dict(payload_b=b, rtt_min_ms=s["min"], rtt_avg_ms=s["avg"],
                     rtt_max_ms=s["max"], mdev_ms=s["mdev"], loss_pct=s["loss_pct"], n=s["n"]))
rtt_sz = pd.DataFrame(rows).sort_values("payload_b")
rtt_sz.to_csv(os.path.join(DATA_OUT, "rtt_vs_pktsize.csv"), index=False)
register("data/rtt_vs_pktsize.csv",
         [f"20260325_000034/raw/ping_size_{b}B.txt" for b in sizes],
         "Parsed min/avg/max/mdev from each ping_size_*B.txt summary line.")

x = rtt_sz["payload_b"].to_numpy(float)
avg = rtt_sz["rtt_avg_ms"].to_numpy(float)
lo = avg - rtt_sz["rtt_min_ms"].to_numpy(float)
hi = rtt_sz["rtt_max_ms"].to_numpy(float) - avg
fig, ax = plt.subplots(figsize=(5.0, 2.8))
ax.errorbar(x, avg, yerr=[lo, hi], color=C_DL, ecolor=C_REF, elinewidth=1.0,
            capsize=3, marker="o", ms=5, lw=1.3, label="avg RTT (min..max whiskers)")
ax.set_xscale("log")
ax.set_xticks(x); ax.set_xticklabels([str(int(v)) for v in x])
ax.set_xlabel("ICMP payload (bytes)"); ax.set_ylabel("RTT (ms)")
ax.set_ylim(0, None); ax.legend(loc="upper left", framealpha=0.9)
save_fig(fig, "fig_rtt_vs_pktsize.pdf")
register("figures/fig_rtt_vs_pktsize.pdf",
         [f"20260325_000034/raw/ping_size_{b}B.txt" for b in sizes],
         "avg RTT vs payload size (log x); whiskers span measured min..max.")

KPI["RTT-vs-size avg (ms)"] = stats(rtt_sz["rtt_avg_ms"])
print(rtt_sz.to_string(index=False))


## Figure 5 - MAVLink drone C2 jitter vs packet rate (OTA run) - HEADLINE

Source: `iperf3_udp_drone_{ul,dl}_{10,50,100}pps.json` (run `20260325_000034`,
50-byte UDP emulating MAVLink). Parsed directly from each file's `end.sum`.
All conditions: 0% packet loss.

In [ ]:
# --- Drone C2: parse 6 iperf3 UDP JSONs -> tidy -> grouped-bar jitter ---
rows = []
for direction in ("ul", "dl"):
    for rate in (10, 50, 100):
        fp = os.path.join(RUN2_RAW, f"iperf3_udp_drone_{direction}_{rate}pps.json")
        if not os.path.exists(fp):
            MISSING.append(f"iperf3_udp_drone_{direction}_{rate}pps.json absent")
            continue
        j = parse_iperf_udp(fp)
        rows.append(dict(rate_pps=rate, direction=direction.upper(),
                         jitter_ms=round(j["jitter_ms"], 4),
                         loss_pct=round(float(j["lost_percent"]), 4),
                         throughput_bps=round(j["bits_per_second"], 1),
                         packets=j["packets"]))
drone = pd.DataFrame(rows).sort_values(["direction", "rate_pps"]).reset_index(drop=True)
drone.to_csv(os.path.join(DATA_OUT, "drone_c2.csv"), index=False)
register("data/drone_c2.csv",
         [f"20260325_000034/raw/iperf3_udp_drone_{d}_{r}pps.json" for d in ("ul","dl") for r in (10,50,100)],
         "Extracted end.sum jitter_ms / lost_percent / bits_per_second / packets per rate & direction.")

rates = [10, 50, 100]
ul_j = [drone[(drone.direction=="UL") & (drone.rate_pps==r)]["jitter_ms"].iloc[0] for r in rates]
dl_j = [drone[(drone.direction=="DL") & (drone.rate_pps==r)]["jitter_ms"].iloc[0] for r in rates]
xi = np.arange(len(rates)); w = 0.38
fig, ax = plt.subplots(figsize=(5.2, 3.0))
b1 = ax.bar(xi - w/2, ul_j, w, color=C_UL, edgecolor="white", lw=0.8, label="UL (UE->gNB)")
b2 = ax.bar(xi + w/2, dl_j, w, color=C_DL, edgecolor="white", lw=0.8, label="DL (gNB->UE)",
            hatch="//")
for bars, vals in ((b1, ul_j), (b2, dl_j)):
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, v+0.2, f"{v:.2f}",
                ha="center", va="bottom", fontsize=7.5)
ax.set_xticks(xi); ax.set_xticklabels([f"{r} pps" for r in rates])
ax.set_xlabel("MAVLink packet rate (50-byte UDP)"); ax.set_ylabel("Jitter (ms)")
ax.set_ylim(0, max(ul_j)*1.22)
ax.legend(loc="upper left", framealpha=0.9)
ax.text(0.98, 0.95, "0% packet loss (all conditions)", transform=ax.transAxes,
        ha="right", va="top", fontsize=8, style="italic",
        bbox=dict(boxstyle="round,pad=0.3", fc="#eef6ec", ec="#7bbf6a", lw=0.8))
save_fig(fig, "fig_drone_c2.pdf")
register("figures/fig_drone_c2.pdf",
         [f"20260325_000034/raw/iperf3_udp_drone_{d}_{r}pps.json" for d in ("ul","dl") for r in (10,50,100)],
         "Grouped bars: UL & DL jitter per packet rate; 0% loss annotated.")

KPI["Drone UL jitter (ms)"] = stats(drone[drone.direction=="UL"]["jitter_ms"])
KPI["Drone DL jitter (ms)"] = stats(drone[drone.direction=="DL"]["jitter_ms"])
KPI["Drone loss (%)"]       = stats(drone["loss_pct"])
print(drone.to_string(index=False))


## Figure 6 - RF / PHY link indicators (coax run)

Source: **REPORT.md Section 2.1** of run `20260505_162512` (gNB log snapshots per test phase).
These values are transcribed **verbatim** from the report table (they are not present as a CSV,
so they are hard-coded here with the report as the cited source, not computed or invented).
Two stacked panels share the phase axis (no dual y-axis): UL SNR (dB) and UL/DL BLER.

In [ ]:
# --- RF/PHY KPIs transcribed verbatim from REPORT.md Section 2.1 (run 20260505_162512) ---
# Phase | UL SNR dB | RSRP dBm | UL MCS | DL MCS | UL BLER | DL BLER
rf_rows = [
    ("Preflight", 22.0, -82.0, "QPSK(3)",  "QPSK(0)",  0.0000, 0.7181),
    ("T1 TCP UL", 11.5, -82.0, "QPSK(8)",  "QPSK(4)",  0.0692, 0.0958),
    ("T2 TCP DL", 23.5, -82.0, "16QAM(10)","QPSK(8)",  0.0334, 0.0999),
    ("T3 bidir",  12.0, -82.0, "QPSK(6)",  "QPSK(6)",  0.1009, 0.1141),
    ("T4 UDP UL", 13.0, -82.0, "QPSK(6)",  "QPSK(2)",  0.0627, 0.0000),
    ("T5 UDP DL", 25.0, -83.0, "QPSK(6)",  "QPSK(5)",  0.1607, 0.0610),
    ("T8 load",   14.0, np.nan,"QPSK(9)",  "16QAM(10)",0.2242, 0.0358),
    ("Final",     14.0, np.nan,"QPSK(9)",  "16QAM(10)",0.2242, 0.0358),
]
rf = pd.DataFrame(rf_rows, columns=["phase","ul_snr_db","rsrp_dbm","ul_mcs","dl_mcs","ul_bler","dl_bler"])
rf.to_csv(os.path.join(DATA_OUT, "rf_phy_kpis.csv"), index=False)
register("data/rf_phy_kpis.csv", ["20260505_162512/REPORT.md (Section 2.1)"],
         "Verbatim transcription of the RF/PHY per-phase table (SNR/RSRP/MCS/BLER) from REPORT.md.")

xi = np.arange(len(rf))
fig, (a1, a2) = plt.subplots(2, 1, figsize=(5.6, 4.0), sharex=True,
                             gridspec_kw={"height_ratios":[1,1], "hspace":0.15})
a1.bar(xi, rf["ul_snr_db"], color=C_DL, edgecolor="white", lw=0.7)
for i, v in enumerate(rf["ul_snr_db"]):
    a1.text(i, v+0.4, f"{v:.1f}", ha="center", va="bottom", fontsize=7)
a1.set_ylabel("UL SNR (dB)"); a1.set_ylim(0, rf["ul_snr_db"].max()*1.18)

w = 0.4
a2.bar(xi-w/2, rf["ul_bler"], w, color=C_UL, edgecolor="white", lw=0.7, label="UL BLER")
a2.bar(xi+w/2, rf["dl_bler"], w, color=C_DL, edgecolor="white", lw=0.7, hatch="//", label="DL BLER")
a2.axhline(0.10, color=C_REF, ls="--", lw=1.0)
a2.text(len(rf)-0.5, 0.11, "0.10 target", color="#666", fontsize=6.5, ha="right")
a2.set_ylabel("BLER"); a2.set_ylim(0, max(rf["ul_bler"].max(), rf["dl_bler"].max())*1.15)
a2.set_xticks(xi); a2.set_xticklabels(rf["phase"], rotation=30, ha="right")
a2.legend(loc="upper center", ncol=2, framealpha=0.9)
save_fig(fig, "fig_rf_phy.pdf")
register("figures/fig_rf_phy.pdf", ["20260505_162512/REPORT.md (Section 2.1)"],
         "Top: UL SNR per phase. Bottom: UL/DL BLER per phase with 0.10 HARQ target line.")

KPI["UL SNR (dB) [phases]"] = stats(rf["ul_snr_db"])
KPI["UL BLER [phases]"]     = stats(rf["ul_bler"])
KPI["DL BLER [phases]"]     = stats(rf["dl_bler"])
print(rf.to_string(index=False))


## Figure 7 - Idle RTT vs distance, n41 (separate fixed testbed, preliminary range context)

Source: `ping_{20,25,30,35,40,45,45boosted}m.txt` (raw per-packet ping logs) plus
`ota_outside_50m/summary.md` (only the summary stats were retained for the 50 m point, no
raw per-packet log - value transcribed verbatim from that file, not computed here).

**Important:** this campaign ran on a *different, non-portable* testbed (fixed x86 hosts
"net"/gNB and "tech"/UE, band n41 2593.35 MHz) than the paired embedded Jetson AGX Orin
nodes (jNET/jUE) that are the subject of this paper. It is included only as preliminary
range context for the frequency-agility discussion, not as a measured property of the
paired-node platform itself.

In [ ]:
# --- Idle RTT vs distance, n41 separate fixed testbed (preliminary range context) ---
N41_DIST_DIR = os.path.join(BASE, "raw_measurements", "n41_distance_sweep_20260513_separate_testbed")

dist_rows = []
dist_sources = []
for dm, folder in [(20,"ota_outside_20m"), (25,"ota_outside_25m"), (30,"ota_outside_30m"),
                    (35,"ota_outside_35m"), (40,"ota_outside_40m"), (45,"ota_outside_45m")]:
    fp = os.path.join(N41_DIST_DIR, folder, f"ping_{dm}m.txt")
    if not os.path.exists(fp):
        MISSING.append(f"ping_{dm}m.txt absent -> distance {dm} m not plotted")
        continue
    s = parse_ping(fp)
    dist_rows.append(dict(distance_m=dm, rtt_min_ms=s["min"], rtt_avg_ms=s["avg"],
                          rtt_max_ms=s["max"], mdev_ms=s["mdev"], loss_pct=s["loss_pct"],
                          n=s["n"], source="raw_ping_log"))
    dist_sources.append(f"n41_distance_sweep_20260513_separate_testbed/{folder}/ping_{dm}m.txt")

# 50 m: only summary.md was retained (no raw per-packet log) - transcribed verbatim, cited.
dist_rows.append(dict(distance_m=50, rtt_min_ms=20.3, rtt_avg_ms=37.0, rtt_max_ms=139.0,
                      mdev_ms=25.2, loss_pct=0.0, n=20, source="summary.md (verbatim)"))
dist_sources.append("n41_distance_sweep_20260513_separate_testbed/ota_outside_50m/summary.md")

dist_rtt = pd.DataFrame(dist_rows).sort_values("distance_m")
dist_rtt.to_csv(os.path.join(DATA_OUT, "rtt_vs_distance_n41_separate_testbed.csv"), index=False)
register("data/rtt_vs_distance_n41_separate_testbed.csv", dist_sources,
         "Per-distance RTT min/avg/max/mdev; 20-45m parsed from raw ping logs, 50m "
         "transcribed verbatim from summary.md (no raw log retained for that point).")

x = dist_rtt["distance_m"].to_numpy(float)
avg = dist_rtt["rtt_avg_ms"].to_numpy(float)
lo = dist_rtt["rtt_min_ms"].to_numpy(float)
hi = dist_rtt["rtt_max_ms"].to_numpy(float)
fig, ax = plt.subplots(figsize=(5.2, 2.9))
ax.fill_between(x, lo, hi, color=C_DL, alpha=0.15, label="min-max range")
ax.plot(x, avg, color=C_DL, marker="o", ms=5, lw=1.5, label="avg RTT")
ax.axhline(20, color=C_REF, ls=":", lw=1.1, label="~20 ms TDD-frame floor")
ax.set_xlabel("Distance (m)"); ax.set_ylabel("ICMP RTT (ms)")
ax.set_xticks(x.astype(int))
ax.legend(loc="upper left", framealpha=0.9)
save_fig(fig, "fig_rtt_vs_distance_n41.pdf")
register("figures/fig_rtt_vs_distance_n41.pdf", dist_sources,
         "Idle ICMP RTT (avg, min-max band) vs distance, n41 separate fixed testbed; "
         "20 ms TDD-frame floor reference line.")

print(dist_rtt.to_string(index=False))


## Provenance table, KPI summary, and missing-data notes

In [ ]:
# --- Provenance table (figure/CSV -> exact source file(s) -> transformation) ---
prov_df = pd.DataFrame([
    {"output": p["output"], "sources": " ; ".join(p["sources"]), "transformation": p["transformation"]}
    for p in PROV
])
pd.set_option("display.max_colwidth", None)
print("="*100); print("PROVENANCE"); print("="*100)
for _, r in prov_df.iterrows():
    print(f"\n[{r['output']}]")
    print(f"  sources : {r['sources']}")
    print(f"  applied : {r['transformation']}")


In [ ]:
# --- KPI summary: real min / mean / max of every plotted series ---
print("="*70); print("KPI SUMMARY (real measured values, for quoting in the paper)"); print("="*70)
sk = pd.DataFrame([{"KPI": k, "min": v["min"], "mean": v["mean"], "max": v["max"], "n": v.get("n")}
                   for k, v in KPI.items()])
with pd.option_context("display.float_format", lambda x: f"{x:.4f}"):
    print(sk.to_string(index=False))


In [ ]:
# --- Missing / not-plotted notes ---
print("="*70); print("MISSING / ESTIMATED NOTES"); print("="*70)
if MISSING:
    for m in MISSING:
        print("  MISSING -", m)
else:
    print("  None. Every deliverable figure was produced from real measured files.")
print("\nNote: no ESTIMATE-labelled figure was produced - all six figures are 100% measured data.")

# --- Confirm all six PDFs exist on disk ---
expected = ["fig_tcp_timeseries.pdf","fig_udp_sweep.pdf","fig_ping_rtt.pdf",
            "fig_rtt_vs_pktsize.pdf","fig_drone_c2.pdf","fig_rf_phy.pdf"]
print("\nFigure files written:")
for f in expected:
    p = os.path.join(FIG_OUT, f)
    print(f"  {'OK ' if os.path.exists(p) else 'MISSING '} {f}  ({os.path.getsize(p) if os.path.exists(p) else 0} bytes)")
